# 📚 Business Analyst Techniques
### 비즈니스 애널리스트 특화

> **Section 11 of 11 (final!)** · Pandas Complete Reference Guide for a JS/TS developer transitioning into BA  
> 전체 11개 섹션 중 **11번째 (마지막!)** · JS/TS 개발자 출신 BA를 위한 Pandas 완전 참조 가이드

---
# 🎯 Learning Objective
Today I want to learn: / 오늘 배울 내용:
- [x] How to compute MoM / YoY growth, and use a cumulative sum for Pareto (80/20) analysis  
MoM / YoY 성장률을 계산하고, 누적합으로 파레토(80/20) 분석을 하는 방법
- [x] How `rank()` differs overall vs within a group, and how to build a basic RFM customer segmentation  
`rank()`가 전체와 그룹 내에서 어떻게 다른지, 기본적인 RFM 고객 세분화를 만드는 방법
- [x] How to flag outliers with the IQR method, write a reusable data quality check, and style a table for a presentation-ready report  
IQR 방식으로 이상치를 표시하고, 재사용 가능한 데이터 품질 체크를 작성하고, 발표용 테이블을 스타일링하는 방법

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

**English**
This final section isn't new pandas syntax so much as it is a toolkit of **named business patterns**, each built entirely from techniques already covered in Sections 1-10: growth rates, an 80/20 breakdown, ranking, customer segmentation, outlier flags, a data health check, and presentation styling. These are the patterns that show up by name in real BA job descriptions and weekly reports.

**한글**
이 마지막 섹션은 새로운 pandas 문법이라기보다는, 1~10번 섹션에서 이미 다룬 기법들로 완전히 만들어진 **이름 붙은 비즈니스 패턴**의 도구 모음입니다: 성장률, 80/20 분석, 순위, 고객 세분화, 이상치 표시, 데이터 상태 점검, 발표용 스타일링. 이는 실제 BA 채용 공고와 주간 보고서에 이름으로 등장하는 패턴들입니다.

## Why do we use it?
*(When is it useful?)*

**English**
A stakeholder rarely asks for "a groupby" — they ask for "month-over-month growth," "our top customers," "which products actually drive revenue." Knowing the underlying pandas mechanics (Sections 1-10) is necessary but not sufficient; this section is the translation layer between that mechanics and the specific questions a business actually asks.

**한글**
이해관계자는 "groupby 해주세요"라고 요청하는 경우가 거의 없습니다 — "전월 대비 성장률", "우리 상위 고객들", "실제로 매출을 이끄는 제품이 뭔지"를 요청합니다. 밑바탕의 pandas 메커니즘(1~10번 섹션)을 아는 것은 필요하지만 충분하지 않습니다. 이 섹션은 그 메커니즘과 비즈니스가 실제로 묻는 구체적인 질문 사이의 번역 계층입니다.

## When is it used in Business Analytics?
*(Real-world use case)*

**English**
Nearly every one of today's 8 patterns appears, by name, in an actual BA/data analyst job posting or a real weekly report — MoM growth in a KPI dashboard, Pareto analysis in inventory planning, RFM in a marketing segmentation project, IQR outlier flags before trusting an average, and a reusable data-quality function that runs first on literally every new dataset.

**한글**
오늘의 8가지 패턴 거의 전부가 실제 BA/데이터 애널리스트 채용 공고나 실제 주간 보고서에 이름으로 등장합니다 — KPI 대시보드의 MoM 성장률, 재고 계획의 파레토 분석, 마케팅 세분화 프로젝트의 RFM, 평균을 신뢰하기 전의 IQR 이상치 표시, 그리고 말 그대로 모든 새 데이터셋에 가장 먼저 실행하는 재사용 가능한 데이터 품질 함수.

### Quick Comparison: Manual vs pandas / 빠른 비교

| Concept / 개념 | Manual approach / 수동 방식 | pandas |
|---|---|---|
| Month-over-month % / 전월 대비 % | `(this - last) / last` per row / 행마다 `(this - last) / last` | `series.pct_change()` |
| Year-over-year % / 전년 동기 대비 % | manually look up "12 rows back" / "12행 전"을 직접 조회 | `series.shift(12)` |
| 80/20 (Pareto) breakdown / 80/20(파레토) 분석 | sort, then sum running totals by hand / 정렬 후 누적 합계를 직접 계산 | `sorted_series.cumsum() / total` |
| Rank within a group / 그룹 내 순위 | sort each group separately, by hand / 각 그룹을 따로 직접 정렬 | `df.groupby("key")["col"].rank()` |
| Outlier detection / 이상치 탐지 | eyeball a chart / 차트를 눈으로 확인 | `Q1 - 1.5*IQR`, `Q3 + 1.5*IQR` fences / 경계값 |
| "Is this data trustworthy?" check / "이 데이터 믿을 만한가?" 확인 | manually inspect a few rows / 몇 행을 직접 검토 | a reusable `data_quality_report(df)` function / 재사용 가능한 함수 |

---
# 📝 Syntax

## Basic Syntax

In [ ]:
import pandas as pd

monthly = pd.DataFrame({
    "month": ["2024-01", "2024-02", "2024-03"],
    "sales": [5678000, 5188000, 5560000],
})

# Month-over-month % change, in one call / 전월 대비 % 변화, 한 번의 호출로
monthly["MoM_pct"] = (monthly["sales"].pct_change() * 100).round(1)
print(monthly)

     month    sales  MoM_pct
0  2024-01  5678000      NaN
1  2024-02  5188000     -8.6
2  2024-03  5560000      7.2


## Common Variations

In [2]:
import pandas as pd

products = pd.DataFrame({
    "product": ["Laptop", "Monitor", "Keyboard", "Mouse"],
    "revenue": [36000000, 17500000, 5400000, 5000000],
})

# Pareto -- sort, then cumulative % of total / 파레토 -- 정렬 후 누적 비중 %
products = products.sort_values("revenue", ascending=False)
products["cum_pct"] = (products["revenue"] / products["revenue"].sum() * 100).cumsum().round(1)
print(products)
print()

# rank -- overall standing / rank -- 전체 순위
products["rank"] = products["revenue"].rank(ascending=False, method="min").astype(int)
print(products)

    product   revenue  cum_pct
0    Laptop  36000000     56.3
1   Monitor  17500000     83.7
2  Keyboard   5400000     92.2
3     Mouse   5000000    100.0

    product   revenue  cum_pct  rank
0    Laptop  36000000     56.3     1
1   Monitor  17500000     83.7     2
2  Keyboard   5400000     92.2     3
3     Mouse   5000000    100.0     4


---
# 🧪 Small Examples

## Example 1 — MoM / YoY: Period-over-Period Comparison
*(Covers source section 11-1)*

**English:** `.pct_change()` computes the % change from the *previous* row in one call — the standard way to get month-over-month growth. For year-over-year, `.shift(12)` (or however many periods make up a year in the data) pulls the value from 12 rows back, so `current / shift(12) - 1` gives the YoY rate.  
**한글:** `.pct_change()`는 한 번의 호출로 *이전* 행 대비 % 변화를 계산합니다 — 전월 대비 성장률을 구하는 표준 방법입니다. 전년 동기 대비의 경우, `.shift(12)`(또는 데이터에서 1년에 해당하는 기간 수)가 12행 전의 값을 가져오므로, `current / shift(12) - 1`로 YoY 비율을 구합니다.

In [ ]:
import pandas as pd

# MoM -- month-over-month, via pct_change() / MoM -- 전월 대비, pct_change() 사용
monthly = pd.DataFrame({
    "month": ["2024-01","2024-02","2024-03","2024-04","2024-05","2024-06"],
    "sales": [5678000, 5188000, 5560000, 5766000, 5431000, 5325000],
})
monthly["MoM_pct"] = (monthly["sales"].pct_change() * 100).round(1)
monthly["MoM_diff"] = monthly["sales"].diff()
print("MoM:")
print(monthly)
print()

# YoY -- year-over-year, via shift(12) on 2 years of monthly data / YoY -- 전년 동기 대비, 2년치 월별 데이터에 shift(12)
sales_2y = pd.DataFrame({
    "month": [f"2023-{str(m).zfill(2)}" for m in range(1, 7)] + [f"2024-{str(m).zfill(2)}" for m in range(1, 7)],
    "sales": [4900000, 4600000, 5100000, 5300000, 4800000, 4700000, 5678000, 5188000, 5560000, 5766000, 5431000, 5325000],
})
sales_2y["YoY_pct"] = ((sales_2y["sales"] / sales_2y["sales"].shift(6) - 1) * 100).round(1)
print("YoY (6-month shift since this sample only spans 2 half-years):")
print(sales_2y.tail(6))

MoM:
     month    sales  MoM_pct  MoM_diff
0  2024-01  5678000      NaN       NaN
1  2024-02  5188000     -8.6 -490000.0
2  2024-03  5560000      7.2  372000.0
3  2024-04  5766000      3.7  206000.0
4  2024-05  5431000     -5.8 -335000.0
5  2024-06  5325000     -2.0 -106000.0

YoY (6-month shift since this sample only spans 2 half-years):
      month    sales  YoY_pct
6   2024-01  5678000     15.9
7   2024-02  5188000     12.8
8   2024-03  5560000      9.0
9   2024-04  5766000      8.8
10  2024-05  5431000     13.1
11  2024-06  5325000     13.3


## Example 2 — Cumulative Sum + Pareto Analysis
*(Covers source section 11-2)*

**English:** Pareto analysis (the "80/20 rule") asks: how few items actually drive most of the total? Sort by value descending, compute each row's % of the grand total, then `.cumsum()` that percentage — the row where the cumulative % first crosses 80% marks the boundary of the "vital few."  
**한글:** 파레토 분석("80/20 법칙")은 묻습니다: 얼마나 적은 항목이 실제로 전체 대부분을 만드는가? 값 기준 내림차순으로 정렬하고, 각 행의 전체 대비 %를 계산한 뒤, 그 비율에 `.cumsum()`을 적용하세요 — 누적 %가 처음으로 80%를 넘는 행이 "핵심 소수"의 경계를 표시합니다.

In [13]:
import pandas as pd

products = pd.DataFrame({
    "product": ["Laptop", "Monitor", "Keyboard", "Mouse", "Webcam", "USB Hub", "Pad", "Speaker"],
    "revenue": [36000000, 17500000, 5400000, 5000000, 4800000, 2800000, 1500000, 900000],
})

products = products.sort_values("revenue", ascending=False).reset_index(drop=True)
products["rev_pct"] = (products["revenue"] / products["revenue"].sum() * 100).round(1)
products["cum_pct"] = products["rev_pct"].cumsum().round(1)
products["item_pct"] = ((products.index + 1) / len(products) * 100).round(1)
print(products)
print()

# Isolate the "vital few" driving 80% of revenue / 매출 80%를 만드는 "핵심 소수" 골라내기
core_products = products[products["cum_pct"] <= 80]
print(f"{len(core_products)} out of {len(products)} products drive 80% of total revenue:")
print(core_products[["product", "revenue", "rev_pct", "cum_pct", "item_pct"]])

    product   revenue  rev_pct  cum_pct  item_pct
0    Laptop  36000000     48.7     48.7      12.5
1   Monitor  17500000     23.7     72.4      25.0
2  Keyboard   5400000      7.3     79.7      37.5
3     Mouse   5000000      6.8     86.5      50.0
4    Webcam   4800000      6.5     93.0      62.5
5   USB Hub   2800000      3.8     96.8      75.0
6       Pad   1500000      2.0     98.8      87.5
7   Speaker    900000      1.2    100.0     100.0

3 out of 8 products drive 80% of total revenue:
    product   revenue  rev_pct  cum_pct  item_pct
0    Laptop  36000000     48.7     48.7      12.5
1   Monitor  17500000     23.7     72.4      25.0
2  Keyboard   5400000      7.3     79.7      37.5


## Example 3 — rank(): Overall vs Within-Group Ranking
*(Covers source section 11-3)*

**English:** `series.rank(ascending=False)` gives an **overall** ranking. `groupby("key")["col"].rank(ascending=False)` gives a ranking **within each group** — someone might be #1 in their department while sitting at #4 company-wide. `method="min"` gives tied values the same (lower) rank; `pct=True` expresses rank as a percentile instead of a position.  
**한글:** `series.rank(ascending=False)`는 **전체** 순위를 줍니다. `groupby("key")["col"].rank(ascending=False)`는 **각 그룹 안에서**의 순위를 줍니다 — 누군가는 부서 내에서 1위이면서 전사에서는 4위일 수 있습니다. `method="min"`은 동점자에게 같은(더 낮은) 순위를 주고, `pct=True`는 순위를 위치 대신 백분위로 표현합니다.

In [5]:
import pandas as pd

df = pd.DataFrame({
    "name": ["Minsu", "Younghee", "Junho", "Seoyeon", "Daehyun", "Soyoung"],
    "dept": ["Sales", "Marketing", "Engineering", "Sales", "Marketing", "Engineering"],
    "sales": [3200000, 2800000, 4500000, 2100000, 3600000, 5100000],
})

df["overall_rank"] = df["sales"].rank(ascending=False, method="min").astype(int)
df["dept_rank"] = df.groupby("dept")["sales"].rank(ascending=False, method="min").astype(int)
df["pct_rank"] = (df["sales"].rank(pct=True) * 100).round(0).astype(int)

print(df.sort_values("overall_rank"))
print()
print("-> Soyoung is #1 both overall AND within Engineering.")
print("-> Daehyun is #3 overall but #1 within Marketing -- rank depends entirely on which group you compare against.")
print("-> Soyoung은 전체에서도 Engineering 안에서도 1위입니다.")
print("-> Daehyun은 전체로는 3위지만 Marketing 안에서는 1위입니다 -- 순위는 어떤 그룹과 비교하는지에 완전히 달려 있습니다.")

       name         dept    sales  overall_rank  dept_rank  pct_rank
5   Soyoung  Engineering  5100000             1          1       100
2     Junho  Engineering  4500000             2          2        83
4   Daehyun    Marketing  3600000             3          1        67
0     Minsu        Sales  3200000             4          1        50
1  Younghee    Marketing  2800000             5          2        33
3   Seoyeon        Sales  2100000             6          2        17

-> Soyoung is #1 both overall AND within Engineering.
-> Daehyun is #3 overall but #1 within Marketing -- rank depends entirely on which group you compare against.
-> Soyoung은 전체에서도 Engineering 안에서도 1위입니다.
-> Daehyun은 전체로는 3위지만 Marketing 안에서는 1위입니다 -- 순위는 어떤 그룹과 비교하는지에 완전히 달려 있습니다.


## Example 4 — RFM Analysis Basics
*(Covers source section 11-4)*

**English:** RFM scores customers on three dimensions computed via `groupby().agg()`: **R**ecency (days since their last order — lower is better), **F**requency (order count), and **M**onetary (total spend). Each dimension gets split into tiers with `pd.qcut()` and combined into a single 3-digit code (like `"333"` for a best-in-every-dimension customer) — a standard customer segmentation used in marketing.  
**한글:** RFM은 `groupby().agg()`로 계산한 세 가지 차원으로 고객을 점수화합니다: **R**ecency(최근 주문 후 경과일 — 낮을수록 좋음), **F**requency(주문 횟수), **M**onetary(총 소비액). 각 차원은 `pd.qcut()`으로 등급으로 나뉜 뒤 하나의 3자리 코드로 결합됩니다(모든 차원에서 최고인 고객은 `"333"`처럼) — 마케팅에서 쓰이는 표준 고객 세분화입니다.

In [6]:
import pandas as pd
import numpy as np

np.random.seed(42)
dates = pd.date_range("2023-01-01", "2024-06-30", freq="D")
orders = pd.DataFrame({
    "customer_id": np.random.choice(["C001", "C002", "C003", "C004", "C005"], 50),
    "order_date": pd.to_datetime(np.random.choice(dates, 50)),
    "revenue": np.random.randint(10000, 200000, 50),
    "order_id": range(1, 51),
})

# Step 1 -- compute R, F, M per customer via groupby().agg() / 1단계 -- groupby().agg()로 고객별 R, F, M 계산
snapshot = pd.Timestamp("2024-07-01")   # the "today" this analysis is run from / 이 분석을 실행하는 "오늘" 기준일
rfm = orders.groupby("customer_id").agg(
    Recency = ("order_date", lambda x: (snapshot - x.max()).days),
    Frequency = ("order_id", "count"),
    Monetary = ("revenue", "sum"),
).reset_index()
print("Step 1 -- raw RFM values:")
print(rfm)
print()

# Step 2 -- score each dimension 1-3 with qcut, then combine into one code
# 2단계 -- qcut으로 각 차원을 1~3점으로 점수화한 뒤 하나의 코드로 결합
rfm["R"] = pd.qcut(rfm["Recency"], q=3, labels=[3, 2, 1])                              # more recent = higher score / 최근일수록 높은 점수
rfm["F"] = pd.qcut(rfm["Frequency"].rank(method="first"), q=3, labels=[1, 2, 3])       # more orders = higher score / 주문이 많을수록 높은 점수
rfm["M"] = pd.qcut(rfm["Monetary"].rank(method="first"), q=3, labels=[1, 2, 3])        # more spend = higher score / 소비가 많을수록 높은 점수
rfm["RFM_Score"] = rfm["R"].astype(str) + rfm["F"].astype(str) + rfm["M"].astype(str)

print("Step 2 -- scored and segmented:")
print(rfm[["customer_id", "Recency", "Frequency", "Monetary", "R", "F", "M", "RFM_Score"]])
print()
print("-> '333' = a recent, frequent, big-spending customer -- prime target for a loyalty campaign.")
print("-> '333' = 최근에, 자주, 많이 쓴 고객 -- 로열티 캠페인의 최우선 타겟.")

Step 1 -- raw RFM values:
  customer_id  Recency  Frequency  Monetary
0        C001       49          7    805529
1        C002       55         10    872713
2        C003       93         10   1241580
3        C004       27         13   1786496
4        C005       39         10   1394201

Step 2 -- scored and segmented:
  customer_id  Recency  Frequency  Monetary  R  F  M RFM_Score
0        C001       49          7    805529  2  1  1       211
1        C002       55         10    872713  1  1  1       111
2        C003       93         10   1241580  1  2  2       122
3        C004       27         13   1786496  3  3  3       333
4        C005       39         10   1394201  3  3  3       333

-> '333' = a recent, frequent, big-spending customer -- prime target for a loyalty campaign.
-> '333' = 최근에, 자주, 많이 쓴 고객 -- 로열티 캠페인의 최우선 타겟.


## Example 5 — Outlier Detection: The IQR Method
*(Covers source section 11-5)*

**English:** The IQR (interquartile range) method finds a statistically "normal" range without assuming a bell curve: `IQR = Q3 - Q1`, and anything below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR` counts as an outlier. Flagging (not deleting) is usually the right first move — a flagged value might be a data-entry error, or it might be a real, unusually large deal worth investigating rather than discarding.  
**한글:** IQR(사분위 범위) 방식은 종형 곡선을 가정하지 않고 통계적으로 "정상" 범위를 찾습니다: `IQR = Q3 - Q1`이며, `Q1 - 1.5*IQR` 미만이거나 `Q3 + 1.5*IQR` 초과인 값은 이상치로 간주됩니다. (삭제가 아니라) 표시해두는 것이 보통 올바른 첫 조치입니다 — 표시된 값은 데이터 입력 오류일 수도 있지만, 버릴 게 아니라 조사해볼 가치가 있는 실제의, 유난히 큰 거래일 수도 있습니다.

In [14]:
import pandas as pd

sales_data = pd.DataFrame({
    "product": ["A","B","C","D","E","F","G","H","I","J"],
    "monthly_sales": [120000, 135000, 98000, 142000, 115000, 850000, 128000, 107000, 5000, 131000],
})

Q1 = sales_data["monthly_sales"].quantile(0.25)
Q3 = sales_data["monthly_sales"].quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

print(f"Q1={Q1:,.0f}  Q3={Q3:,.0f}  IQR={IQR:,.0f}")
print(f"normal range: {lower_fence:,.0f} ~ {upper_fence:,.0f}")
print()

outliers = sales_data[(sales_data["monthly_sales"] < lower_fence) | (sales_data["monthly_sales"] > upper_fence)]
print(f"outliers: {len(outliers)} of {len(sales_data)}")
print(outliers)
print()

# Flagging, not deleting -- each one needs a different follow-up / 삭제가 아니라 표시 -- 각각 다른 후속 조치가 필요
print("Product F (850,000, ~5x the upper fence) -- likely a genuine large one-off deal, worth investigating, not discarding.")
print("Product I (5,000, below the lower fence) -- likely a data entry error or a discontinued item.")
print("Product F(850,000, 상한의 약 5배) -- 실제 대형 일회성 거래일 가능성, 버리지 말고 조사할 가치가 있음.")
print("Product I(5,000, 하한 미만) -- 데이터 입력 오류이거나 단종된 제품일 가능성.")

Q1=109,000  Q3=134,000  IQR=25,000
normal range: 71,500 ~ 171,500

outliers: 2 of 10
  product  monthly_sales
5       F         850000
8       I           5000

Product F (850,000, ~5x the upper fence) -- likely a genuine large one-off deal, worth investigating, not discarding.
Product I (5,000, below the lower fence) -- likely a data entry error or a discontinued item.
Product F(850,000, 상한의 약 5배) -- 실제 대형 일회성 거래일 가능성, 버리지 말고 조사할 가치가 있음.
Product I(5,000, 하한 미만) -- 데이터 입력 오류이거나 단종된 제품일 가능성.


## Example 6 — A Reusable Data Quality Check Function
*(Covers source section 11-6)*

**English:** This function packages several checks from Sections 3 and 5 — dtypes, missing-value %, uniqueness (to spot ID vs category columns, just like Section 3), a sample value per column, and duplicate row count — into a single reusable call. Running it first on every new dataset is the fastest way to know whether it can be trusted.

⚠️ **The duplicate-check trap:** `df.duplicated()` compares **every** column, so a single surrogate key column (`id`, `order_id`) makes it structurally always `0` — two identical records entered twice with different IDs will never be flagged. That's why this version takes an `id_cols=` argument and reports **both** counts: all-columns and business-columns-only.

**한글:** 이 함수는 3번과 5번 섹션의 여러 확인 — dtype, 결측치 %, 고유성(3번 섹션처럼 ID 열과 카테고리 열을 구분하기 위함), 열별 샘플 값, 중복 행 개수 — 을 하나의 재사용 가능한 호출로 묶습니다. 새로운 데이터셋마다 가장 먼저 이를 실행하는 것이 신뢰할 수 있는지 아는 가장 빠른 방법입니다.

⚠️ **중복 검사의 함정:** `df.duplicated()`는 **모든** 열을 비교하므로, `id` / `order_id` 같은 대리키 열이 하나라도 있으면 결과가 구조적으로 항상 `0`이 됩니다 — ID만 다른 동일 레코드는 절대 잡히지 않습니다. 그래서 이 버전은 `id_cols=` 인자를 받아 **두 가지** 개수(전체 열 기준 / 비즈니스 열 기준)를 모두 보고합니다.

In [17]:
import pandas as pd

def data_quality_report(df, id_cols=None):
    """Summarize a DataFrame's quality in one glance.

    id_cols: surrogate key columns (order_id, customer_id, ...) to exclude from
             the duplicate check. A column with a unique value per row makes
             df.duplicated() structurally always 0.
    id_cols: 중복 검사에서 제외할 대리키 열. 행마다 값이 다른 열이 하나라도
             포함되면 df.duplicated()는 구조적으로 항상 0이 됨.
    """
    report = pd.DataFrame({
        "dtype": df.dtypes,
        "null_cnt": df.isna().sum(),
        "null_pct": (df.isna().sum() / len(df) * 100).round(1),
        "unique": df.nunique(),
        "unique_pct": (df.nunique() / len(df) * 100).round(1),
        "sample": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })

    id_cols = id_cols or []
    business_cols = [c for c in df.columns if c not in id_cols]
    dup_all = df.duplicated().sum()
    dup_business = df.duplicated(subset=business_cols).sum() if business_cols else 0
    id_candidates = [c for c in business_cols if df[c].nunique() == len(df)]

    print(f"Shape: {df.shape}")
    print(f"Duplicate rows (all columns):          {dup_all}")
    print(f"Duplicate rows (excluding {id_cols if id_cols else 'nothing'}):  {dup_business}")
    if id_candidates:
        print(f"[!] unique_pct == 100% -> consider id_cols={id_candidates}")
    return report

# Try it on a deliberately messy sample / 일부러 지저분하게 만든 샘플에 적용
test_df = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "name": ["A", "B", None, "D", "A"],
    "score": [85, 90, None, 78, 85],
    "category": ["X", "Y", "X", "Y", "X"],
})

# Without id_cols -- the ID column hides the duplicate / id_cols 없이 -- ID 열이 중복을 숨김
print("=== id_cols 없이 ===")
print(data_quality_report(test_df))
print()

# With id_cols -- rows 0 and 4 are the same record entered twice
# id_cols 지정 -- 0번과 4번 행은 같은 레코드가 두 번 입력된 것
print("=== id_cols=['id'] ===")
print(data_quality_report(test_df, id_cols=["id"]))
print()
print("Reading it / 읽는 법:")
print("- id: unique_pct == 100% -> an ID column, not a groupby candidate / ID 열이지 groupby 대상이 아님")
print("- category: unique_pct is low -> THIS is a groupby candidate / groupby 후보")
print("- name, score: both missing 1 value (20%) -> worth a closer look before analysis / 분석 전에 자세히 볼 가치가 있음")
print("- 'all columns' dup == 0 but 'excluding id' dup == 1 -> an ID-only difference, i.e. the same")
print("  record entered twice / 전체 기준 0인데 id 제외 기준 1 -> ID만 다른 중복 입력")
print("- sample: the first real value per column -> catches an object column holding dates/lists")
print("  / 열별 실제 값 하나 -> object 열에 날짜나 리스트가 들어있는 경우를 잡아냄")

=== id_cols 없이 ===
Shape: (5, 4)
Duplicate rows (all columns):          0
Duplicate rows (excluding nothing):  0
[!] unique_pct == 100% -> consider id_cols=['id']
            dtype  null_cnt  null_pct  unique  unique_pct sample
id          int64         0       0.0       5       100.0      1
name          str         1      20.0       3        60.0      A
score     float64         1      20.0       3        60.0   85.0
category      str         0       0.0       2        40.0      X

=== id_cols=['id'] ===
Shape: (5, 4)
Duplicate rows (all columns):          0
Duplicate rows (excluding ['id']):  1
            dtype  null_cnt  null_pct  unique  unique_pct sample
id          int64         0       0.0       5       100.0      1
name          str         1      20.0       3        60.0      A
score     float64         1      20.0       3        60.0   85.0
category      str         0       0.0       2        40.0      X

Reading it / 읽는 법:
- id: unique_pct == 100% -> an ID column, not a gr

## Example 7 — Styling for Presentation-Ready Tables
*(Covers source section 11-7)*

**English:** `.style` turns a DataFrame into a `Styler` object that renders as a formatted HTML table — `.format()` controls number display (commas, %, decimals), `.background_gradient()` colors cells by magnitude (a built-in heatmap), and `.bar()` draws an inline bar inside each cell, like Excel's "Data Bars." ⚠️ **This only renders visually in a Jupyter/Colab notebook** — running the same code in a plain `.py` script produces no visible formatting.  
**한글:** `.style`는 DataFrame을 서식이 적용된 HTML 테이블로 렌더링되는 `Styler` 객체로 바꿉니다 — `.format()`은 숫자 표시(쉼표, %, 소수점)를 제어하고, `.background_gradient()`는 크기에 따라 셀에 색을 입히며(내장 히트맵), `.bar()`는 Excel의 "데이터 막대"처럼 각 셀 안에 인라인 막대를 그립니다. ⚠️ **이는 Jupyter/Colab 노트북에서만 시각적으로 렌더링됩니다** — 같은 코드를 일반 `.py` 스크립트에서 실행하면 눈에 보이는 서식이 나타나지 않습니다.

In [ ]:
import pandas as pd

report = pd.DataFrame({
    "region": ["Seoul", "Busan", "Incheon"],
    "sales": [5678000, 3244000, 2890000],
    "growth": [15.2, -3.4, 8.7],
    "share": [0.482, 0.276, 0.242],
})

# .format() -- control number display / .format() -- 숫자 표시 제어
# NOTE: in Jupyter/Colab, leave `styled` as the LAST line of a cell (no print()) to see it rendered
# 참고: Jupyter/Colab에서는 셀의 마지막 줄에 `styled`만(print() 없이) 남겨두면 렌더링된 모습을 볼 수 있음
styled = report.style.format({
    "sales": "{:,.0f}",     # thousands separator / 천단위 구분 쉼표
    "growth": "{:+.1f}%",   # signed percentage / 부호 포함 퍼센트
    "share": "{:.1%}",      # ratio -> percentage / 비율 -> 퍼센트
})
print("Styler object built -- in Jupyter, write 'styled' alone on the last line to render it:")
print(type(styled))
print()

# .background_gradient() -- a built-in heatmap / .background_gradient() -- 내장 히트맵
styled_heatmap = (
    report.style
    .format({"sales": "{:,.0f}", "growth": "{:+.1f}%"})
    .background_gradient(subset=["sales"], cmap="Blues")
    .background_gradient(subset=["growth"], cmap="RdYlGn")
)
print("Heatmap Styler built:", type(styled_heatmap))
print()

# .bar() -- an inline bar per cell, like Excel's Data Bars / .bar() -- 셀 안 인라인 막대, Excel 데이터 막대와 유사
summary = pd.DataFrame({
    "product": ["Laptop", "Monitor", "Keyboard"],
    "revenue": [36000000, 17500000, 5400000],
    "margin": [25.0, 28.6, 33.3],
})
styled_bars = (
    summary.style
    .format({"revenue": "{:,.0f}", "margin": "{:.1f}%"})
    .bar(subset=["revenue"], color="#5B9BD5")
    .bar(subset=["margin"], color="#70AD47")
)
print("Data-bar Styler built:", type(styled_bars))

Styler object built -- in Jupyter, write 'styled' alone on the last line to render it:
<class 'pandas.io.formats.style.Styler'>

Heatmap Styler built: <class 'pandas.io.formats.style.Styler'>

Data-bar Styler built: <class 'pandas.io.formats.style.Styler'> <pandas.io.formats.style.Styler object at 0x1117d1100>


## Example 8 — Common Combo: BA Report Patterns
*(Covers source section 11-8)*

**English:** Pattern A builds a full monthly-by-region report in one chain: `assign()` for a month label, `groupby()` for regional totals, and `groupby().rank()` for each region's standing within its month. Pattern B chains sorting, `assign()`, `cumsum()`, and `query()` into a one-shot Pareto filter — straight from raw data to "here are the vital few."   
**한글:** 패턴 A는 한 번의 체인으로 전체 월별×지역별 보고서를 만듭니다: 월 라벨을 위한 `assign()`, 지역별 합계를 위한 `groupby()`, 각 지역의 월 내 순위를 위한 `groupby().rank()`. 패턴 B는 정렬, `assign()`, `cumsum()`, `query()`를 체이닝해서 한 번에 파레토 필터를 만듭니다 — 원시 데이터에서 곧바로 "이것이 핵심 소수입니다"까지.

In [21]:
import pandas as pd
import numpy as np

# Pattern A: a full monthly-by-region report in one chain
# 패턴 A: 한 번의 체인으로 만드는 전체 월별x지역별 보고서
np.random.seed(9)
daily = pd.DataFrame({
    "date": pd.date_range("2024-01-01", "2024-06-30", freq="D"),
    "region": np.random.choice(["Seoul", "Busan", "Incheon"], 182),
    "sales": np.random.randint(50000, 300000, 182),
})

monthly = (
    daily
    .assign(month=daily["date"].dt.strftime("%Y-%m"))
    .groupby(["month", "region"])["sales"]
    .sum()
    .reset_index()
)
monthly["rank_in_month"] = monthly.groupby("month")["sales"].rank(ascending=False, method="min").astype(int)
print("Pattern A -- monthly x region report with rank:")
print(monthly.head(9))
print()

# Pattern B: raw data straight to a Pareto filter, one chain
# 패턴 B: 원시 데이터에서 한 번의 체인으로 파레토 필터까지
products = pd.DataFrame({
    "product": ["Laptop", "Monitor", "Keyboard", "Mouse", "Webcam", "USB Hub", "Pad", "Speaker"],
    "revenue": [36000000, 17500000, 5400000, 5000000, 4800000, 2800000, 1500000, 900000],
})
core = (
    products
    .sort_values("revenue", ascending=False)
    .reset_index(drop=True)
    .assign(
        rev_pct = lambda x: (x["revenue"] / x["revenue"].sum() * 100).round(1),
        cum_pct = lambda x: (x["revenue"] / x["revenue"].sum() * 100).cumsum().round(1),
    )
    .query("cum_pct <= 80")
)
print("Pattern B -- straight to the vital few:")
print(core)

Pattern A -- monthly x region report with rank:
     month   region    sales  rank_in_month
0  2024-01    Busan  1329228              3
1  2024-01  Incheon  1756952              2
2  2024-01    Seoul  1892239              1
3  2024-02    Busan  1222143              3
4  2024-02  Incheon  1669933              2
5  2024-02    Seoul  2191597              1
6  2024-03    Busan  1022076              3
7  2024-03  Incheon  1857736              1
8  2024-03    Seoul  1809458              2

Pattern B -- straight to the vital few:
    product   revenue  rev_pct  cum_pct
0    Laptop  36000000     48.7     48.7
1   Monitor  17500000     23.7     72.4
2  Keyboard   5400000      7.3     79.7


## Example 9 (Practice) — Fill in the Blanks
*(Based on the practice exercise in source section 11-9 -- a capstone combining Section 10's time series with this section's techniques)*

**English:** Fill in each `________` blank below. The code is syntactically valid Python, so it won't raise a `SyntaxError` — but it also won't print any result until every blank is correct (it will raise a runtime error instead, which is expected).  
**한글:** 아래 `________` 빈칸을 채워보세요. 코드는 문법적으로 올바른 파이썬이라 `SyntaxError`는 나지 않지만, 모든 빈칸이 정확해지기 전까지는 결과가 출력되지 않습니다(대신 런타임 오류가 나는데, 이는 의도된 동작입니다).

In [22]:
import pandas as pd
import numpy as np

np.random.seed(7)
daily = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=90, freq="D"),
    "region": np.random.choice(["Seoul", "Busan", "Incheon"], 90),
    "sales": np.random.randint(50000, 300000, 90),
})

# 1. Add a "month" column as a YYYY-MM string / "month" 열을 YYYY-MM 문자열로 추가
daily["month"] = daily["date"].dt.strftime("%Y-%m")

# 2. Aggregate to monthly totals, by month AND region / 월 x 지역별 합계로 집계
monthly = daily.groupby(["month", "region"])["sales"].sum().reset_index()

# 3. Add each region's MoM % change (compared to its OWN prior month)
#    각 지역의 MoM % 변화 추가 (자기 자신의 이전 달과 비교) 
monthly["MoM_pct"] = (monthly.groupby("region")["sales"].pct_change() * 100).round(1)

# 4. Add each region's rank WITHIN its month / 각 지역의 월 내 순위 추가
monthly["rank_in_month"] = monthly.groupby("month")["sales"].rank(ascending=False, method="min").astype(int)

print(monthly)

     month   region    sales  MoM_pct  rank_in_month
0  2024-01    Busan   879217      NaN              3
1  2024-01  Incheon  1728257      NaN              2
2  2024-01    Seoul  2501561      NaN              1
3  2024-02    Busan  1885195    114.4              2
4  2024-02  Incheon  1113561    -35.6              3
5  2024-02    Seoul  2336607     -6.6              1
6  2024-03    Busan  1543846    -18.1              2
7  2024-03  Incheon  1296801     16.5              3
8  2024-03    Seoul  1664363    -28.8              1


### 💡 Hint / 힌트
`strftime` · `groupby` · `pct_change` · `rank`

### ✅ Solution / 정답
*(Try solving it yourself first! / 먼저 스스로 풀어본 뒤에 확인하세요!)*

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(7)
daily = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=90, freq="D"),
    "region": np.random.choice(["Seoul", "Busan", "Incheon"], 90),
    "sales": np.random.randint(50000, 300000, 90),
})

daily["month"] = daily["date"].dt.strftime("%Y-%m")
monthly = daily.groupby(["month", "region"])["sales"].sum().reset_index()
monthly["MoM_pct"] = (monthly.groupby("region")["sales"].pct_change() * 100).round(1)
monthly["rank_in_month"] = monthly.groupby("month")["sales"].rank(ascending=False, method="min").astype(int)

print(monthly)

# January has no MoM_pct for any region (no prior month to compare against) -- expected.
# Seoul ranks #1 every single month here -- worth flagging as a consistent top performer
# in an actual report, not dismissing as a coincidence of this particular random sample.
# 1월은 어느 지역이든 MoM_pct가 없음(비교할 이전 달이 없어서) -- 정상입니다.
# Seoul은 매달 1위를 차지하는데 -- 실제 보고서라면 이 무작위 샘플의 우연으로 넘기지 말고
# 꾸준한 최고 실적자로 표시할 가치가 있습니다.

---
# ⚠️ Common Mistakes

### Mistake 1 — Deleting IQR-flagged outliers by default
**English:** An outlier might be the single most important row in the dataset — a genuinely huge deal, not an error. Deleting it automatically because it crossed a statistical fence can hide the most interesting story in the data instead of cleaning it.  
**한글:** 이상치는 데이터셋에서 가장 중요한 단일 행일 수 있습니다 — 오류가 아니라 진짜로 거대한 거래일 수 있습니다. 통계적 경계를 넘었다는 이유로 자동으로 삭제하면, 데이터를 정제하는 게 아니라 가장 흥미로운 이야기를 숨기는 것일 수 있습니다.

**✅ Fix / 해결법:**  
Always flag first, investigate second, and only remove a value once you've confirmed it's actually an error — not just statistically unusual.  
항상 먼저 표시하고, 그다음 조사하고, 실제로 오류라고 확인한 뒤에만 제거하세요 — 단지 통계적으로 특이한 것만으로는 부족합니다.

### Mistake 2 — Assuming `shift(12)` always means "one year ago"
**English:** `shift(12)` assumes exactly 12 rows equal one year, which silently breaks if the data has a gap (a missing month) or isn't monthly to begin with — weekly data would need `shift(52)`, not `shift(12)`.  
**한글:** `shift(12)`는 정확히 12행이 1년과 같다고 가정하는데, 데이터에 빈틈이 있거나(누락된 달) 애초에 월별이 아니라면 조용히 틀립니다 — 주별 데이터라면 `shift(12)`가 아니라 `shift(52)`가 필요합니다.

**✅ Fix / 해결법:**  
Before using a fixed `shift(n)`, confirm the data has no gaps and matches the period you're assuming (monthly, weekly, etc.) — a quick `.shape` and date-range check catches this.  
고정된 `shift(n)`을 쓰기 전에, 데이터에 빈틈이 없고 가정한 기간(월별, 주별 등)과 일치하는지 확인하세요 — 빠른 `.shape`와 날짜 범위 확인으로 잡을 수 있습니다.

### Mistake 3 — Expecting `.style` output to show up outside a notebook
**English:** `df.style.format(...).background_gradient(...)` produces a `Styler` object — HTML under the hood. `print()`-ing it, running it in a terminal, or saving it in a plain `.py` script shows no formatting at all, which can look like the code silently failed.  
**한글:** `df.style.format(...).background_gradient(...)`는 `Styler` 객체를 만듭니다 — 내부적으로는 HTML입니다. 이를 `print()`하거나, 터미널에서 실행하거나, 일반 `.py` 스크립트에 저장하면 서식이 전혀 보이지 않는데, 코드가 조용히 실패한 것처럼 보일 수 있습니다.

**✅ Fix / 해결법:**  
Only expect `.style` formatting to render inside a Jupyter/Colab cell, with the styled object as the last unprinted line. For anything that needs to leave the notebook (an email, a saved file), export with `to_excel()` and format there instead.  
`.style` 서식은 Jupyter/Colab 셀 안에서, 스타일 객체가 출력 없이 마지막 줄에 있을 때만 렌더링된다고 생각하세요. 노트북 밖으로 나가야 하는 것(이메일, 저장 파일)은 `to_excel()`로 내보내서 거기서 서식을 적용하세요.

---
# 💡 Tips
Useful tips or shortcuts / 유용한 팁과 단축법

- Flag outliers, don't reflexively delete them — the IQR fence tells you *where* to look, not automatically *what* to do.  
이상치는 표시하되 반사적으로 삭제하지 마세요 — IQR 경계는 *어디를* 봐야 할지 알려줄 뿐, *무엇을* 해야 할지 자동으로 알려주지 않습니다.
- Run the reusable `data_quality_report()` function (Example 6) as the very first thing on any new dataset — it's a 3-line habit that catches most of Sections 3 and 5's lessons in one call.  
새 데이터셋에는 항상 가장 먼저 재사용 가능한 `data_quality_report()` 함수(Example 6)를 실행하세요 — 3줄짜리 습관으로 3번과 5번 섹션의 교훈 대부분을 한 번에 잡아냅니다.
- `.style` chains only render in Jupyter/Colab — if the same formatted look needs to leave the notebook (an emailed report, a saved file), export to Excel via `to_excel()` and format there instead.  
`.style` 체인은 Jupyter/Colab에서만 렌더링됩니다 — 같은 서식이 노트북 밖으로 나가야 한다면(이메일 보고서, 저장 파일) `to_excel()`로 내보내서 거기서 서식을 적용하세요.
- MoM / YoY / rank / Pareto are all just Sections 1-10's tools with a business name attached — if a pattern here feels unfamiliar, the fix is almost always to revisit the specific earlier section it's built from.  
MoM / YoY / rank / 파레토는 전부 1~10번 섹션의 도구에 비즈니스 이름을 붙인 것뿐입니다 — 여기서 어떤 패턴이 낯설게 느껴진다면, 그 패턴이 세워진 구체적인 이전 섹션을 다시 보는 것이 거의 항상 해결책입니다.

---
# 🔗 Related Concepts

```
Time Series                     (Section 10 -- shift()/resample() are the literal mechanics behind today's MoM/YoY)
    ↓
Business Analyst Techniques       ← you are here / 지금 여기 (Section 11, final)
    ↓
... back to Section 1 -- everything here rests on the Series/DataFrame foundation from day one
```

*How does this final topic connect to the rest of the series?*

**English:** Nothing in this notebook is really "new." MoM/YoY is Section 10's `shift()` and `pct_change()`. Pareto is Section 6's `sort_values()` plus a cumulative sum. `rank()` builds on Section 7's GroupBy. RFM is `groupby().agg()` (Section 7) combined with `pd.qcut()` (Section 6). The IQR method is `.quantile()` and Boolean indexing from Section 4. The data quality function packages Section 3's EDA checks and Section 5's cleaning checks into one call. And styling is a presentation layer on top of everything. If any example here felt shaky, that's a signal for which earlier section is worth a second pass — not a sign you need to learn something entirely new. That's really the whole point of this 11-section series: a small, fixed set of pandas mechanics, recombined in different orders to answer whatever question the business is actually asking.

**한글:** 이 노트북에서 진짜로 "새로운" 것은 없습니다. MoM/YoY는 10번 섹션의 `shift()`와 `pct_change()`입니다. 파레토는 6번 섹션의 `sort_values()`와 누적합입니다. `rank()`는 7번 섹션의 GroupBy 위에 세워집니다. RFM은 `groupby().agg()`(7번 섹션)와 `pd.qcut()`(6번 섹션)의 결합입니다. IQR 방식은 4번 섹션의 `.quantile()`과 Boolean indexing입니다. 데이터 품질 함수는 3번 섹션의 EDA 확인과 5번 섹션의 정제 확인을 하나의 호출로 묶은 것입니다. 그리고 스타일링은 이 모든 것 위에 얹힌 발표 계층입니다. 이 중 어떤 예제라도 낯설게 느껴졌다면, 그건 완전히 새로운 걸 배워야 한다는 신호가 아니라 어느 이전 섹션을 다시 한번 보면 좋을지 알려주는 신호입니다. 이것이 바로 이 11개 섹션 시리즈 전체의 요점입니다 — 작고 고정된 pandas 기법 집합을, 비즈니스가 실제로 묻는 질문에 답하기 위해 다른 순서로 재조합하는 것입니다.

---
# 💼 Business Example
*How would a Business Analyst use this?*

**Scenario / 시나리오**

**English:** It's time for a quarterly business review. Before presenting anything, run a data quality check on the raw order data, then build the monthly revenue trend with MoM%, and identify which products actually drive most of the revenue.

**한글:** 분기 사업 리뷰 시간입니다. 무엇이든 발표하기 전에, 원본 주문 데이터에 데이터 품질 확인을 실행하고, MoM%가 포함된 월별 매출 추세를 만들고, 실제로 매출 대부분을 이끄는 제품이 무엇인지 파악하세요.

**To Do / 할 일**
- [x] Run the data quality check on the raw data first, always  
원본 데이터에 항상 가장 먼저 데이터 품질 확인 실행하기
- [x] Build the monthly revenue trend with MoM%  
MoM%가 포함된 월별 매출 추세 만들기
- [x] Identify the products driving most of the revenue with a Pareto breakdown  
파레토 분석으로 매출 대부분을 이끄는 제품 파악하기

In [23]:
import pandas as pd
import numpy as np

np.random.seed(11)
dates_q1 = pd.date_range("2024-01-01", "2024-03-31", freq="D")
orders = pd.DataFrame({
    "order_date": np.random.choice(dates_q1, 150),
    "product": np.random.choice(["Laptop", "Monitor", "Keyboard", "Mouse"], 150),
    "revenue": np.random.randint(20000, 500000, 150),
})

# Step 1 -- data quality check FIRST, always / 1단계 -- 항상 가장 먼저 데이터 품질 확인
def data_quality_report(df):
    report = pd.DataFrame({
        "dtype": df.dtypes,
        "null_pct": (df.isna().sum() / len(df) * 100).round(1),
        "unique": df.nunique(),
    })
    print(f"Shape: {df.shape} | Duplicate rows: {df.duplicated().sum()}")
    return report

print("Data quality check:")
print(data_quality_report(orders))
print()

# Step 2 -- monthly revenue trend with MoM% / 2단계 -- MoM%가 포함된 월별 매출 추세
orders["month"] = orders["order_date"].dt.strftime("%Y-%m")
monthly = orders.groupby("month")["revenue"].sum().reset_index()
monthly["MoM_pct"] = (monthly["revenue"].pct_change() * 100).round(1)
print("Monthly trend:")
print(monthly)
print()

# Step 3 -- which products actually drive revenue? (Pareto) / 3단계 -- 실제로 매출을 이끄는 제품은? (파레토)
by_product = (
    orders.groupby("product")["revenue"].sum()
    .reset_index()
    .sort_values("revenue", ascending=False)
    .assign(cum_pct=lambda x: (x["revenue"] / x["revenue"].sum() * 100).cumsum().round(1))
)
print("Product Pareto:")
print(by_product)

Data quality check:
Shape: (150, 3) | Duplicate rows: 0
                     dtype  null_pct  unique
order_date  datetime64[us]       0.0      67
product                str       0.0       4
revenue              int64       0.0     150

Monthly trend:
     month   revenue  MoM_pct
0  2024-01  11354344      NaN
1  2024-02  12648135     11.4
2  2024-03  11572603     -8.5

Product Pareto:
    product   revenue  cum_pct
3     Mouse  10855750     30.5
0  Keyboard   9200814     56.4
2   Monitor   8426225     80.1
1    Laptop   7092293    100.0


---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

**English**
This final section is a toolkit of named business patterns built entirely from Sections 1-10's mechanics. `pct_change()` and `shift()` compute MoM and YoY growth; sorting plus `cumsum()` isolates the "vital few" items driving 80% of a total (Pareto); `rank()` — especially `groupby().rank()` — distinguishes overall standing from standing within a group; RFM scores customers on Recency/Frequency/Monetary via `groupby().agg()` plus `pd.qcut()`; the IQR method (`Q1`, `Q3`, and 1.5× the interquartile range) flags outliers for investigation rather than automatic deletion; a reusable `data_quality_report()` function packages dtype, missing-value, and uniqueness checks into one call; and `.style` formatting turns a plain DataFrame into a presentation-ready table inside a notebook. Every one of these patterns is really just earlier chapters, recombined with a business question attached.

**한글**
이 마지막 섹션은 1~10번 섹션의 기법으로 완전히 만들어진 이름 붙은 비즈니스 패턴의 도구 모음입니다. `pct_change()`와 `shift()`는 MoM과 YoY 성장률을 계산하고, 정렬과 `cumsum()`은 전체의 80%를 만드는 "핵심 소수"를 골라내며(파레토), `rank()` — 특히 `groupby().rank()` — 는 전체 순위와 그룹 내 순위를 구분합니다. RFM은 `groupby().agg()`와 `pd.qcut()`으로 고객을 Recency/Frequency/Monetary로 점수화하고, IQR 방식(`Q1`, `Q3`, 사분위 범위의 1.5배)은 자동 삭제 대신 조사를 위해 이상치를 표시하며, 재사용 가능한 `data_quality_report()` 함수는 dtype, 결측치, 고유성 확인을 하나의 호출로 묶고, `.style` 서식은 순수 DataFrame을 노트북 안에서 발표 가능한 테이블로 바꿉니다. 이 모든 패턴은 사실 이전 챕터들이 비즈니스 질문과 함께 재조합된 것일 뿐입니다.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence.

> Every technique in this final section — MoM/YoY, Pareto, rank, RFM, outlier detection, data quality checks, styling — is Sections 1 through 10's pandas mechanics wearing a business name, which is the real lesson of this entire 11-section guide: a small set of tools, recombined endlessly to answer whatever the business actually asks.

> 이 마지막 섹션의 모든 기법 — MoM/YoY, 파레토, rank, RFM, 이상치 탐지, 데이터 품질 확인, 스타일링 — 은 비즈니스 이름을 붙인 1~10번 섹션의 pandas 기법이며, 이것이 바로 이 11개 섹션 전체 가이드의 진짜 교훈입니다 — 작은 도구 집합을 끝없이 재조합해서 비즈니스가 실제로 묻는 것에 답하는 것입니다.

---
# ❓ Review Questions

**Q1.** What's the difference between what `pct_change()` and `shift(12)` each compute, and which one gives you YoY?  
**Q1.** `pct_change()`와 `shift(12)`가 각각 계산하는 것의 차이는 무엇이며, 어느 것이 YoY를 주나요?

shift(12) retrieves the value from 12 periods ago, while pct_change() calculates the percentage change from the previous value. YoY is usually calculated by combining shift(12) with pct_change().  
shift(12)는 12개월 전의 값을 가져오는 것이고, pct_change()는 현재 값과 이전 값의 변화율(%)을 계산. YoY는 보통 shift(12)와 pct_change()를 함께 사용해서 계산.

**Q2.** In a Pareto analysis, what does the row where `cum_pct` first exceeds 80% actually tell you?  
**Q2.** 파레토 분석에서, `cum_pct`가 처음으로 80%를 넘는 행은 실제로 무엇을 알려주나요?

The first row where cum_pct exceeds 80% tells you how many top items are needed to account for roughly 80% of the total value.  
cum_pct가 처음으로 80%를 넘는 행은 전체 매출(또는 해당 지표)의 누적 80%를 만들기 위해 필요한 상위 항목의 경계를 알려줌.

**Q3.** Why might the same employee have a different rank when you compute `rank()` overall vs. `groupby("dept").rank()`?  
**Q3.** 같은 직원이 `rank()`를 전체로 계산했을 때와 `groupby("dept").rank()`로 계산했을 때 왜 다른 순위를 가질 수 있나요?

Overall rank() compares all employees together, while groupby("dept").rank() compares employees only within their department.  
전체 rank()는 모든 직원을 서로 비교하지만, groupby("dept").rank()는 같은 부서 안에서만 직원들을 비교하기 때문에 순위가 달라질 수 있음.

**Q4.** In IQR outlier detection, what should you do first when you find a flagged value — delete it, or something else?  
**Q4.** IQR 이상치 탐지에서, 표시된 값을 발견했을 때 가장 먼저 무엇을 해야 하나요 — 삭제인가요, 아니면 다른 것인가요?

When an IQR outlier is detected, do not delete it immediately; first investigate why it is an outlier.  
IQR 이상치가 발견되면 바로 삭제하지 말고, 먼저 그 값이 왜 이상한지 확인.

**Q5.** Why does a `.style`-formatted table look perfect in a Jupyter notebook but show no formatting at all when the same code runs in a plain `.py` script?  
**Q5.** `.style`로 서식을 입힌 테이블이 Jupyter 노트북에서는 완벽해 보이는데, 같은 코드가 일반 `.py` 스크립트에서 실행되면 왜 서식이 전혀 안 보이나요?

.style formats a DataFrame for display using HTML/CSS, so Jupyter can render that formatting, while a normal .py script does not automatically display the styled HTML table.  
.style은 DataFrame의 데이터를 변경하는 것이 아니라 HTML/CSS로 표시되는 표현 방식을 만드는 기능이기 때문.

---
*📅 Try answering these again in a few days. / 며칠 뒤에 다시 답해보세요.*

---
### 🎉 That's the full 11-section series / 전체 11개 섹션 시리즈 완주
*From `pd.Series([1,2,3])` in Section 1 to a quarterly business review in Section 11 — every tool along the way was built from the one before it.*  
*1번 섹션의 `pd.Series([1,2,3])`부터 11번 섹션의 분기 사업 리뷰까지 — 그 과정의 모든 도구는 바로 앞의 것 위에 세워졌습니다.*